# STEP03 — MOTH EXP045-equivalent WIDTH512 @50k

Первый длинный training-stage для `ilPloInte3.2`.

Зафиксированный рецепт:
- WIDTH512, RF=1029;
- target=1024, context_flank=130;
- batch=4;
- presence BCE + 0.1 × Huber(log1p(count));
- 0→25k: LR=1e-3;
- 25k→50k: cosine 3e-4→3e-5;
- seed=42;
- один полный natural validation scan только после 50k.

Validation не используется для обучения, mining или выбора checkpoint.


In [1]:
from pathlib import Path
import copy, gc, hashlib, json, math, os, sys, time, warnings
import numpy as np
import pandas as pd
import torch
import torch_directml
from IPython.display import display

EXPECTED_PYTHON = Path(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
).resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (
    sys.executable, EXPECTED_PYTHON
)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (current, *current.parents)
    if (p/"README.md").is_file()
    and (p/"src/ml_project").is_dir()
    and (p/"data/raw/unpacked").is_dir()
)

AGENTS = next(
    (p for p in (PROJECT_ROOT/"AGENTS.md", PROJECT_ROOT.parent/"AGENTS.md")
     if p.is_file()),
    None,
)
assert AGENTS is not None
EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
agents_sha = hashlib.sha256(AGENTS.read_bytes()).hexdigest()
assert agents_sha == EXPECTED_AGENTS_SHA, (agents_sha, EXPECTED_AGENTS_SHA)

LOADER = PROJECT_ROOT/"src/ml_project/epic_data.py"
EXPECTED_LOADER_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"
loader_sha = hashlib.sha256(LOADER.read_bytes()).hexdigest()
assert loader_sha == EXPECTED_LOADER_SHA, (loader_sha, EXPECTED_LOADER_SHA)

SRC = PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    evaluate_presence_intensity_model,
    forward_both_strands_multitask,
)
from ml_project.epic_data import SplitConfig, load_split

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("GPU:", torch_directml.device_name(0))


Project: D:\Projects\EPIC\EPIC
Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
GPU: AMD Radeon RX 6750 GRE 12GB 


In [2]:
ASSEMBLY = "ilPloInte3.2"
SPLIT_CONFIG = SplitConfig(
    assembly=ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

split = load_split(PROJECT_ROOT, SPLIT_CONFIG, verify_hashes=True)
display(split.summary)

train_row = split.summary.loc[split.summary["split"]=="train"].iloc[0]
val_row = split.summary.loc[split.summary["split"]=="validation"].iloc[0]
test_row = split.summary.loc[split.summary["split"]=="test"].iloc[0]

assert int(train_row["contigs"]) == 19
assert int(val_row["contigs"]) == 7
assert int(test_row["contigs"]) == 7
assert int(train_row["positive_positions"]) == 581_052
assert int(val_row["positive_positions"]) == 162_306
assert split.positive_targets("test") is None

sequences, fasta_alphabet = load_baseline_sequences(split)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

probe = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
).sample_batch(4)

assert tuple(probe.x.shape) == (4,5,1284)
assert tuple(probe.x_both_strands.shape) == (8,5,1284)
assert tuple(probe.target.shape) == (4,2,1024)
del probe
gc.collect()

device = torch_directml.device()
model_probe = create_cnn_presence_intensity(channels=512)
n_params = sum(p.numel() for p in model_probe.parameters())
assert n_params == 14_708_738
del model_probe
gc.collect()

print("Readiness: PASS")


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,19,178622,164558039,329116078,60.145242,581052.0,328535026.0,0.176549
1,validation,7,63150,52057100,104114200,19.026642,162306.0,103951894.0,0.155892
2,test,7,67434,56985954,113971908,20.828116,NaN,NaN,NaN


Readiness: PASS


In [3]:
EXPERIMENT_ID = "SUBMIT-MOTH-EXP045"
EXPERIMENT_SLUG = "SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029"

SEED = 42
STEPS = 50_000
PHASE1_END = 25_000
BATCH_SIZE = 4
LOG_EVERY = 250
CHECKPOINT_STEPS = {10_000,25_000,30_000,35_000,40_000,45_000,50_000}

PHASE1_LR = 1e-3
PHASE2_START_LR = 3e-4
PHASE2_END_LR = 3e-5
PHASE2_STEPS = STEPS - PHASE1_END
INTENSITY_WEIGHT = 0.1

RUN_DIR = PROJECT_ROOT/"artifacts/experiments"/EXPERIMENT_SLUG/"run_001"
RUN_DIR.mkdir(parents=True, exist_ok=True)

FINAL = RUN_DIR/"width512_50k_final.pt"
BRANCH = RUN_DIR/"model_step_50000.pt"
HISTORY = RUN_DIR/"training_history.csv"
METRICS = RUN_DIR/"validation_metrics_report.csv"
SUMMARY = RUN_DIR/"result_summary.json"

def model_state_on_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def tree_to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k:tree_to_cpu(v) for k,v in x.items()}
    if isinstance(x, list):
        return [tree_to_cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple(tree_to_cpu(v) for v in x)
    return x

def ckpt_path(step):
    return RUN_DIR/f"model_step_{step:05d}.pt"

def atomic_torch_save(path, payload):
    tmp = path.with_suffix(path.suffix+".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def save_ckpt(path, step, model, opt, sched, gen, history, seen, elapsed):
    payload = {
        "experiment": EXPERIMENT_ID,
        "assembly": ASSEMBLY,
        "model_type": "presence_intensity",
        "architecture": "rf1029_width512",
        "step": int(step),
        "model_state_dict": model_state_on_cpu(model),
        "optimizer_state_dict": tree_to_cpu(opt.state_dict()),
        "scheduler_state_dict": None if sched is None else tree_to_cpu(sched.state_dict()),
        "generator_rng_state": copy.deepcopy(gen.rng.bit_generator.state),
        "torch_rng_state": torch.get_rng_state(),
        "model_config": {
            "input_channels":5, "channels":512, "output_channels":2,
            "theoretical_rf":1029,
        },
        "profile_config": {
            "target_length":1024,
            "context_flank":130,
            "positive_branch_fraction":0.5,
        },
        "training_config": {
            "seed":SEED,
            "steps":STEPS,
            "batch_size":BATCH_SIZE,
            "phase1_end_step":PHASE1_END,
            "phase1_learning_rate":PHASE1_LR,
            "phase2_start_lr":PHASE2_START_LR,
            "phase2_end_lr":PHASE2_END_LR,
            "phase2_scheduler":"CosineAnnealingLR",
            "intensity_loss_weight":INTENSITY_WEIGHT,
            "huber_delta":1.0,
        },
        "loader_sha256": loader_sha,
        "seen_position_strand": int(seen),
        "elapsed_seconds": float(elapsed),
        "history": list(history),
    }
    atomic_torch_save(path, payload)

print("Artifacts:", RUN_DIR)


Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001


## Long training 50k

Можно повторно запустить после прерывания: продолжит с последнего checkpoint.


In [ ]:
if FINAL.is_file():
    done = torch.load(FINAL, map_location="cpu", weights_only=False)
    assert int(done["step"]) == 50_000
    assert done["assembly"] == ASSEMBLY
    print("Already complete:", FINAL)
    del done

else:
    torch.manual_seed(SEED)

    generator = ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=SEED,
    )
    model = create_cnn_presence_intensity(channels=512).to(device)
    optimizer = DirectMLAdam(model.parameters(), lr=PHASE1_LR)
    scheduler = None

    available = [(s, ckpt_path(s)) for s in sorted(CHECKPOINT_STEPS)
                 if ckpt_path(s).is_file()]
    start_step, resume_path = available[-1] if available else (0, None)

    history = []
    seen = 0
    previous_elapsed = 0.0

    if resume_path is not None:
        resume = torch.load(resume_path, map_location="cpu", weights_only=False)
        assert int(resume["step"]) == start_step
        assert resume["assembly"] == ASSEMBLY
        model.load_state_dict(resume["model_state_dict"])
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(resume["generator_rng_state"])
        torch.set_rng_state(resume["torch_rng_state"])
        history = list(resume.get("history", []))
        seen = int(resume.get("seen_position_strand", 0))
        previous_elapsed = float(resume.get("elapsed_seconds", 0.0))

        if start_step < PHASE1_END:
            for g in optimizer.param_groups:
                g["lr"] = PHASE1_LR
        elif start_step == PHASE1_END:
            for g in optimizer.param_groups:
                g["lr"] = PHASE2_START_LR
                g["initial_lr"] = PHASE2_START_LR
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=PHASE2_STEPS, eta_min=PHASE2_END_LR
            )
        else:
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=PHASE2_STEPS, eta_min=PHASE2_END_LR
            )
            scheduler.load_state_dict(resume["scheduler_state_dict"])
        del resume
        gc.collect()
        print("RESUME:", resume_path)

    ema_p = history[-1]["ema_presence"] if history else None
    ema_i = history[-1]["ema_intensity"] if history else None
    ema_t = history[-1]["ema_total"] if history else None

    started = time.perf_counter()
    model.train()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")

        for step in range(start_step+1, STEPS+1):
            batch = generator.sample_batch(BATCH_SIZE)

            x = batch.x_both_strands.to(device)
            pt = batch.plus_target.to(device)
            mt = batch.minus_target_reverse.to(device)
            pw = batch.plus_loss_weight.to(device)
            mw = batch.minus_loss_weight_reverse.to(device)

            pc = batch.count[:,0:1,:].contiguous()
            mc = batch.count[:,1:2,:].flip(-1).contiguous()

            pit = torch.log1p(pc).to(device)
            mit = torch.log1p(mc).to(device)
            piw = (2.0*batch.plus_loss_weight*(pc>0).float()).to(device)
            miw = (2.0*batch.minus_loss_weight_reverse*(mc>0).float()).to(device)

            optimizer.zero_grad(set_to_none=True)

            pp, mp, pi, mi = forward_both_strands_multitask(
                model, x, batch_size=BATCH_SIZE,
                context_flank=130, target_length=1024
            )

            ploss = dml_weighted_profile_bce_with_logits(pp, mp, pt, mt, pw, mw)
            iloss = (
                dml_weighted_huber(pi, pit, piw, delta=1.0)
                + dml_weighted_huber(mi, mit, miw, delta=1.0)
            )
            loss = ploss + INTENSITY_WEIGHT*iloss

            pv = float(ploss.detach().cpu().item())
            iv = float(iloss.detach().cpu().item())
            tv = float(loss.detach().cpu().item())
            assert all(math.isfinite(v) for v in (pv,iv,tv))

            loss.backward()
            optimizer.step()

            # Switch after completing step 25k at LR=1e-3.
            if step == PHASE1_END:
                for g in optimizer.param_groups:
                    g["lr"] = PHASE2_START_LR
                    g["initial_lr"] = PHASE2_START_LR
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=PHASE2_STEPS, eta_min=PHASE2_END_LR
                )
            elif step > PHASE1_END:
                scheduler.step()

            ema_p = pv if ema_p is None else 0.99*ema_p + 0.01*pv
            ema_i = iv if ema_i is None else 0.99*ema_i + 0.01*iv
            ema_t = tv if ema_t is None else 0.99*ema_t + 0.01*tv
            seen += int(batch.mask.sum())

            elapsed_process = time.perf_counter()-started
            elapsed_total = previous_elapsed+elapsed_process
            sec_step = elapsed_process/max(step-start_step,1)

            if step % LOG_EVERY == 0 or step == STEPS:
                rec = {
                    "step":step,
                    "learning_rate":float(optimizer.param_groups[0]["lr"]),
                    "presence_loss":pv,
                    "intensity_loss":iv,
                    "total_loss":tv,
                    "ema_presence":float(ema_p),
                    "ema_intensity":float(ema_i),
                    "ema_total":float(ema_t),
                    "elapsed_seconds":float(elapsed_total),
                    "seconds_per_step_current_process":float(sec_step),
                    "estimated_training_hours_remaining":float((STEPS-step)*sec_step/3600),
                    "seen_position_strand":int(seen),
                }
                history.append(rec)
                pd.DataFrame(history).to_csv(HISTORY,index=False)
                print(
                    f"step={step:05d}/{STEPS} "
                    f"lr={optimizer.param_groups[0]['lr']:.8f} "
                    f"loss=({pv:.4f},{iv:.4f},{tv:.4f}) "
                    f"sec/step={sec_step:.3f} "
                    f"ETA={(STEPS-step)*sec_step/3600:.2f}h",
                    flush=True
                )

            if step in CHECKPOINT_STEPS:
                save_ckpt(
                    ckpt_path(step), step, model, optimizer, scheduler,
                    generator, history, seen, elapsed_total
                )

            del batch,x,pt,mt,pw,mw,pc,mc,pit,mit,piw,miw,pp,mp,pi,mi,ploss,iloss,loss

        fallback = [str(w.message) for w in caught if "fall back" in str(w.message).lower()]
        assert not fallback, fallback

    elapsed_total = previous_elapsed + (time.perf_counter()-started)
    assert scheduler is not None
    assert math.isclose(
        float(optimizer.param_groups[0]["lr"]),
        PHASE2_END_LR,
        rel_tol=1e-6,
        abs_tol=1e-10,
    )

    save_ckpt(FINAL, STEPS, model, optimizer, scheduler, generator,
              history, seen, elapsed_total)

    payload = torch.load(FINAL, map_location="cpu", weights_only=False)
    atomic_torch_save(BRANCH, payload)
    del payload

    print("TRAINING COMPLETE")
    print("Hours:", round(elapsed_total/3600,3))
    print("Branch point:", BRANCH)

    del model, optimizer, scheduler, generator
    gc.collect()


step=00250/50000 lr=0.00100000 loss=(0.8645,0.1594,0.8805) sec/step=0.237 ETA=3.28h
step=00500/50000 lr=0.00100000 loss=(0.5726,0.0694,0.5796) sec/step=0.233 ETA=3.21h
step=00750/50000 lr=0.00100000 loss=(0.8697,0.5064,0.9203) sec/step=0.232 ETA=3.18h
step=01000/50000 lr=0.00100000 loss=(0.6287,0.4409,0.6728) sec/step=0.231 ETA=3.15h
step=01250/50000 lr=0.00100000 loss=(0.6130,0.3676,0.6497) sec/step=0.231 ETA=3.12h
step=01500/50000 lr=0.00100000 loss=(0.8932,0.2171,0.9149) sec/step=0.230 ETA=3.10h
step=01750/50000 lr=0.00100000 loss=(0.6262,0.5129,0.6775) sec/step=0.230 ETA=3.08h
step=02000/50000 lr=0.00100000 loss=(0.5534,0.5164,0.6050) sec/step=0.230 ETA=3.06h
step=02250/50000 lr=0.00100000 loss=(0.5586,0.2072,0.5793) sec/step=0.230 ETA=3.05h
step=02500/50000 lr=0.00100000 loss=(0.4876,0.5979,0.5474) sec/step=0.230 ETA=3.03h
step=02750/50000 lr=0.00100000 loss=(0.5008,0.6347,0.5643) sec/step=0.229 ETA=3.01h
step=03000/50000 lr=0.00100000 loss=(0.4270,0.3049,0.4575) sec/step=0.229 ET

In [ ]:
checkpoint = torch.load(BRANCH, map_location="cpu", weights_only=False)
assert checkpoint["assembly"] == ASSEMBLY
assert int(checkpoint["step"]) == 50_000
branch_sha = hashlib.sha256(BRANCH.read_bytes()).hexdigest()

model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

val_gen = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

result = evaluate_presence_intensity_model(
    model,
    val_gen,
    device=device,
    batch_size=1,
    score_decimals=5,
    progress_every_batches=100,
    progress=print,
    fail_on_cpu_fallback=True,
)

display(result.metrics)
result.metrics.to_csv(METRICS,index=False)

overall = result.metrics.loc[
    (result.metrics["score"]=="presence_probability")
    & (result.metrics["segment"]=="overall")
].iloc[0]

summary = {
    "experiment":EXPERIMENT_ID,
    "assembly":ASSEMBLY,
    "checkpoint":str(BRANCH),
    "checkpoint_sha256":branch_sha,
    "step":50_000,
    "validation_position_strand":int(result.processed_positions),
    "validation_positives":int(result.processed_positives),
    "presence_average_precision_5dp":float(overall["average_precision"]),
    "presence_epic_spearman":float(overall["epic_spearman"]),
    "validation_elapsed_seconds":float(result.elapsed_seconds),
    "validation_positions_per_second":float(result.positions_per_second),
    "fallback_warnings":list(result.fallback_warnings),
    "next_stage":"train-only EXP047 risk cache, then EXP047-B +20k",
}
SUMMARY.write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))

assert result.processed_positions == 104_114_200
assert result.processed_positives == 162_306
assert not result.fallback_warnings

del checkpoint,model,val_gen,result
gc.collect()


tiles=  2100/53710 positions=  4047348 positions/s=81,572
tiles=  2200/53710 positions=  4242086 positions/s=81,618
tiles=  4600/53710 positions=  9011430 positions/s=82,870
tiles=  4700/53710 positions=  9211036 positions/s=82,915
tiles=  7200/53710 positions= 14186082 positions/s=83,328
tiles=  7300/53710 positions= 14384246 positions/s=83,339
tiles=  9700/53710 positions= 19112110 positions/s=83,347
tiles=  9800/53710 positions= 19301496 positions/s=83,314
tiles= 12200/53710 positions= 23917158 positions/s=82,922
tiles= 12300/53710 positions= 24110324 positions/s=82,913
tiles= 12400/53710 positions= 24299074 positions/s=82,892
tiles= 14800/53710 positions= 28997358 positions/s=82,883
tiles= 14900/53710 positions= 29197420 positions/s=82,889
tiles= 17300/53710 positions= 33969446 positions/s=83,111
tiles= 17400/53710 positions= 34169824 positions/s=83,122
tiles= 19900/53710 positions= 39131536 positions/s=83,221
tiles= 20000/53710 positions= 39327760 positions/s=83,222
tiles= 22400/5